In [ ]:
# ========================= IMPORTS =========================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from PIL import Image
from tqdm import tqdm
import timm
import matplotlib.pyplot as plt
import seaborn as sns

# ========================= CONFIG =========================
DATA_DIR = "/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset"
BATCH_SIZE = 16
IMG_SIZE = 224
EPOCHS = 12
LR_HEAD = 1e-4
LR_BACKBONE = 1e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LABELS = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
LABEL_MAP = {l:i for i,l in enumerate(LABELS)}

# ========================= BUILD MANIFEST =========================
def build_manifest(base_dir):
    rows = []
    for disease in os.listdir(base_dir):
        img_dir = os.path.join(base_dir, disease, "images")
        if not os.path.exists(img_dir):
            continue
        for f in os.listdir(img_dir):
            if f.lower().endswith(('.png','.jpg','.jpeg')):
                rows.append([os.path.join(img_dir,f), disease])
    return pd.DataFrame(rows, columns=["path","label"])

df = build_manifest(DATA_DIR)
df = df[df["label"].isin(LABELS)].reset_index(drop=True)

# ========================= SPLIT =========================
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df["label"], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.15, stratify=train_df["label"], random_state=42)

# ========================= TRANSFORMS =========================
train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE,IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# ========================= DATASET =========================
class CXRDataset(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True)
        self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.loc[idx,"path"]).convert("RGB")
        img = self.tf(img)
        label = LABEL_MAP[self.df.loc[idx,"label"]]
        return img, label

train_loader = DataLoader(CXRDataset(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(CXRDataset(val_df, eval_tf), batch_size=BATCH_SIZE)
test_loader  = DataLoader(CXRDataset(test_df, eval_tf), batch_size=BATCH_SIZE)

# ========================= MODEL =========================
model = timm.create_model("densenet121", pretrained=True, num_classes=4)
for p in model.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True
model.to(DEVICE)

# ========================= CLASS WEIGHTS =========================
counts = train_df["label"].value_counts().to_dict()
weights = torch.tensor([1/counts[l] for l in LABELS]).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=LR_HEAD)

# ========================= TRAIN =========================
best_auc = 0
for epoch in range(EPOCHS):
    model.train()
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), labels)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            out = model(imgs.to(DEVICE))
            preds.append(out.softmax(1).cpu().numpy())
            trues.append(labels.numpy())

    preds, trues = np.concatenate(preds), np.concatenate(trues)
    acc = accuracy_score(trues, preds.argmax(1))
    auc = roc_auc_score(pd.get_dummies(trues), preds, multi_class="ovr")
    print(f"Val Acc: {acc:.4f} | AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), "best_model.pth")

# ========================= TEST METRICS =========================
model.load_state_dict(torch.load("best_model.pth"))
model.eval()
preds, trues = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        out = model(imgs.to(DEVICE))
        preds.append(out.softmax(1).cpu().numpy())
        trues.append(labels.numpy())

preds, trues = np.concatenate(preds), np.concatenate(trues)

print("\nClassification Report:\n")
print(classification_report(trues, preds.argmax(1), target_names=LABELS))

cm = confusion_matrix(trues, preds.argmax(1))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS)
plt.savefig("confusion_matrix.png", dpi=300)
plt.show()

# ========================= EXPORT FOR STREAMLIT =========================
example = torch.randn(1,3,224,224).to(DEVICE)
scripted = torch.jit.trace(model, example)
scripted.save("chestxray_model.pt")
